# 00 — Colab Setup


## 1. Mount Drive and create the folder layout

```
MyDrive/crypto-ner/
├── code/      utils.py + the four notebooks
├── data/      Dataset CSVs and LKB JSONs
├── ckpt/      base model checkpoints
├── oof/       out-of-fold predictions
└── analysis/  every table destined for the paper
```

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/crypto-ner'
for sub in ['code', 'data', 'ckpt', 'oof', 'analysis']:
    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)

print('Folder layout ready:')
for sub in sorted(os.listdir(ROOT)):
    n = len(os.listdir(f'{ROOT}/{sub}')) if os.path.isdir(f'{ROOT}/{sub}') else 0
    print(f'  {ROOT}/{sub}  ({n} files)')

Mounted at /content/drive
Folder layout ready:
  /content/drive/MyDrive/crypto-ner/analysis  (0 files)
  /content/drive/MyDrive/crypto-ner/ckpt  (0 files)
  /content/drive/MyDrive/crypto-ner/code  (2 files)
  /content/drive/MyDrive/crypto-ner/data  (4 files)
  /content/drive/MyDrive/crypto-ner/oof  (0 files)


## 2. Install dependencies

`pytorch-crf` is the only unusual one. Colab already ships torch, transformers,
datasets and scikit-learn; the versions it ships are fine.

Ignore any pip dependency-resolver warnings about unrelated Colab packages —
they do not affect this code.

In [ ]:
!pip install -q pytorch-crf seqeval safetensors

import torch, transformers, sklearn
print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('sklearn      :', sklearn.__version__)
print('GPU          :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  <-- switch to a GPU runtime before notebook 02')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 879.4 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
torch        : 2.11.0+cpu
transformers : 5.13.1
sklearn      : 1.6.1
GPU          : NONE  <-- switch to a GPU runtime before notebook 02


## 3. Put `utils.py` where the notebooks can import it

Upload `utils.py` (and `test_utils.py`) to `MyDrive/crypto-ner/code/` once, then
this cell copies them into the working directory each session.

If you have not uploaded them yet, run the fallback cell below to upload from
your machine.

In [ ]:
CODE = f'{ROOT}/code'

if os.path.exists(f'{CODE}/utils.py'):
    for f in ['utils.py', 'test_utils.py']:
        if os.path.exists(f'{CODE}/{f}'):
            shutil.copy(f'{CODE}/{f}', f'/content/{f}')
    print('Copied utils.py from Drive -> /content')
else:
    print('utils.py not found in Drive. Run the upload cell below.')

Copied utils.py from Drive -> /content


In [ ]:
# Fallback: upload utils.py / test_utils.py straight from your computer
from google.colab import files
up = files.upload()                       # select utils.py (and test_utils.py)
for name in up:
    shutil.copy(f'/content/{name}', f'{CODE}/{name}')
    print(f'saved -> {CODE}/{name}')

KeyboardInterrupt: 

## 4. Verify the module works

`test_utils.py` checks entity extraction, the exact-match metric, fold
reproducibility, the denoising heuristic, both significance tests, Fleiss' kappa
and the BiLSTM-CRF. It takes about 30 seconds.

If anything fails here, stop — do not run the experiments.

In [ ]:
import sys
sys.path.append('/content')
!cd /content && python test_utils.py

  PASS  extract_entities
  PASS  entity_f1 (exact match, partial span counts as wrong)
  PASS  count_bio_violations
  PASS  get_folds (reproducible, disjoint; train sizes [1168, 1169, 1169, 1169, 1169])
  PASS  gazetteer denoising suppresses ambiguous tokens; naive does not
  PASS  gazetteer cashtag handling
  PASS  paired_bootstrap (dF1=0.331, p=0.0000) and randomization
  PASS  bootstrap_ci [0.605, 0.731]
  PASS  holm_bonferroni [0.004, 0.06, 0.08, 0.5]
  PASS  fleiss_kappa=0.6571, pairwise entity F1 {'A1-A2': 1.0, 'A1-A3': 0.667, 'A2-A3': 0.667}
  PASS  corpus_statistics
  PASS  assign_strata ['Blue-Chip', 'Blue-Chip', 'Background']
  PASS  TokenClassifier heads produce finite loss and well-formed decodes
  PASS  train_bilstm_crf fits toy data (F1=1.000) and returns aligned predictions

All checks passed.


## 5. Check the data files

Upload these into `MyDrive/crypto-ner/data/`. The Drive web interface is the
easiest route for the larger CSVs.

In [ ]:
DATA = f'{ROOT}/data'

required = {
    'gold_standard_bio_labeled.csv' : 'nb 01, 02, 03',
    'silver_standard_bio.csv'       : 'nb 01 (silver stats)',
    'silver_standard_dataset.csv'   : 'nb 01 (builds the naive silver set)',
    'lkb_cleaned.json'              : 'nb 01, 02, 03',
    'lkb_case_sensitive.json'       : 'nb 01, 02, 03',
    'dataset_mentah_twitter.csv'    : 'nb 01 (preprocessing funnel only)',
}
generated = {
    'silver_standard_bio_naive.csv' : 'generated by nb 01 section D',
}

missing = []
for f, use in required.items():
    here = os.path.exists(f'{DATA}/{f}')
    size = f'{os.path.getsize(f"{DATA}/{f}")/1e6:.1f} MB' if here else '--'
    print(f'  [{"ok " if here else "MISSING"}] {f:34s} {size:>10s}   {use}')
    if not here:
        missing.append(f)

print()
for f, use in generated.items():
    here = os.path.exists(f'{DATA}/{f}')
    print(f'  [{"ok " if here else "pending"}] {f:34s}              {use}')

if missing:
    print(f'\nUpload these to {DATA} before continuing:')
    for f in missing:
        print(f'  - {f}')

  [ok ] gold_standard_bio_labeled.csv          0.9 MB   nb 01, 02, 03
  [ok ] silver_standard_bio.csv               17.5 MB   nb 01 (silver stats)
  [ok ] silver_standard_dataset.csv           17.2 MB   nb 01 (builds the naive silver set)
  [ok ] lkb_cleaned.json                       0.2 MB   nb 01, 02, 03
  [ok ] lkb_case_sensitive.json                0.0 MB   nb 01, 02, 03
  [ok ] dataset_mentah_twitter.csv            16.4 MB   nb 01 (preprocessing funnel only)

  [pending] silver_standard_bio_naive.csv                   generated by nb 01 section D


## 6. Locate the existing base model checkpoint

The proposed system fine-tunes from the base model behind DOI
`10.57967/hf/9017` — the one your original pipeline trained on denoised silver.

Its state-dict keys (`bert.*`, `classifier.*`, `crf.*`) match `TokenClassifier`
exactly, so it loads without conversion. The cell below searches Drive for it.

In [ ]:
import glob

candidates = []
for pat in ['**/indobertweet-crf-base-final', '**/indobertweet-crf-base*',
            '**/indobertweet-crf-PRODUCTION-FINAL']:
    candidates += glob.glob(f'/content/drive/MyDrive/{pat}', recursive=True)

candidates = sorted(set(candidates))
if candidates:
    print('Found checkpoint directories:')
    for c in candidates:
        weights = [f for f in os.listdir(c)
                   if f in ('model.safetensors', 'pytorch_model.bin')]
        print(f'  {c}')
        print(f'     weights: {weights if weights else "NONE — not a usable checkpoint"}')
else:
    print('No base checkpoint found under MyDrive.')
    print('Either copy it into ckpt/, or set RETRAIN_DENOISED_BASE = True in nb 02')
    print('to train a fresh one on the denoised silver set.')

Found checkpoint directories:
  /content/drive/MyDrive/ResearchMethod_AllFiles/Model/indobertweet-crf-PRODUCTION-FINAL
     weights: ['pytorch_model.bin']
  /content/drive/MyDrive/ResearchMethod_AllFiles/Model/indobertweet-crf-base
     weights: NONE — not a usable checkpoint
  /content/drive/MyDrive/ResearchMethod_AllFiles/Model/indobertweet-crf-base-final
     weights: ['model.safetensors']
  /content/drive/MyDrive/indobertweet-crf-PRODUCTION-FINAL
     weights: ['pytorch_model.bin']
  /content/drive/MyDrive/indobertweet-crf-base
     weights: NONE — not a usable checkpoint
  /content/drive/MyDrive/indobertweet-crf-base-final
     weights: ['model.safetensors']


In [ ]:
# Confirm the checkpoint actually loads into TokenClassifier before nb 02 runs
BASE_DENOISED = f'{ROOT}/ckpt/indobertweet-crf-base-final'   # <- adjust if needed

try:
    from utils import TokenClassifier, load_state_dict_any
    sd = load_state_dict_any(BASE_DENOISED)
    m = TokenClassifier('indolem/indobertweet-base-uncased', use_crf=True)
    missing_k, unexpected_k = m.load_state_dict(sd, strict=False)
    print(f'Loaded. missing keys: {len(missing_k)}, unexpected keys: {len(unexpected_k)}')
    if missing_k:
        print('  missing   :', list(missing_k)[:8])
    if unexpected_k:
        print('  unexpected:', list(unexpected_k)[:8])
    if not missing_k and not unexpected_k:
        print('Perfect key match — safe to use as init_from.')
    del m
except FileNotFoundError as e:
    print('Checkpoint not at that path yet:', e)

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded. missing keys: 0, unexpected keys: 0
Perfect key match — safe to use as init_from.


## 7. Paths to paste into notebooks 01–03

Each notebook has a path block near the top that currently points at `/content`.
Replace those three lines with the values printed below, so that checkpoints,
OOF predictions and analysis tables all land on Drive and survive a disconnect.

In [ ]:
print('Paste into notebook 01:')
print(f"    DATA = '{ROOT}/data'")
print(f"    OUT  = '{ROOT}/analysis'")
print()
print('Paste into notebook 02:')
print(f"    DATA = '{ROOT}/data'")
print(f"    OOF  = '{ROOT}/oof'")
print(f"    CKPT = '{ROOT}/ckpt'")
print()
print('Paste into notebook 03:')
print(f"    DATA = '{ROOT}/data'")
print(f"    OOF  = '{ROOT}/oof'")
print(f"    OUT  = '{ROOT}/analysis'")
print()
print("Also add   sys.path.append('/content')   if the import of utils fails.")

Paste into notebook 01:
    DATA = '/content/drive/MyDrive/crypto-ner/data'
    OUT  = '/content/drive/MyDrive/crypto-ner/analysis'

Paste into notebook 02:
    DATA = '/content/drive/MyDrive/crypto-ner/data'
    OOF  = '/content/drive/MyDrive/crypto-ner/oof'
    CKPT = '/content/drive/MyDrive/crypto-ner/ckpt'

Paste into notebook 03:
    DATA = '/content/drive/MyDrive/crypto-ner/data'
    OOF  = '/content/drive/MyDrive/crypto-ner/oof'
    OUT  = '/content/drive/MyDrive/crypto-ner/analysis'

Also add   sys.path.append('/content')   if the import of utils fails.


## 8. Runtime expectations on a T4

| Step | Time |
|---|---|
| nb 01 — everything | 3–8 min (CPU is fine) |
| nb 02 — naive-silver base model | 20–40 min |
| nb 02 — one transformer config (5 folds) | 4–8 min |
| nb 02 — six transformer configs | 30–50 min |
| nb 02 — gazetteer | seconds |
| nb 02 — BiLSTM-CRF (5 folds) | 3–6 min |
| nb 03 — everything | 1–3 min |

Budget roughly **1.5–2 hours of GPU time** for notebook 02 in total.

### Surviving a disconnect

Notebook 02 skips any configuration whose OOF file already exists, so after a
drop you just re-run the setup notebook and the run cell — finished
configurations are not repeated. This only works if `OOF` points at Drive.

Two habits that help: keep the tab visible rather than backgrounded, and run the
expensive base-model cell first so the long job is behind you.

### If you hit the free-tier GPU limit

Notebooks 01 and 03 need no GPU at all — switch to a CPU runtime for those and
save the GPU quota for notebook 02. Within notebook 02, run the configurations
in priority order (`proposed`, `naive_silver_ft`, `indobertweet_crf_gold`,
`indobert_crf_gold`, `base_only`, `softmax_ft`) so that a quota cutoff costs you
the least important rows.

### If you run out of GPU memory

Halve `per_device_train_batch_size` in the affected config (32 → 16 for base
training, 16 → 8 for fine-tuning) and note the change in the paper. Restarting
the runtime between heavy cells also helps, since fragmentation accumulates.